# 01 Strategy Baseline

Purpose:
- Run baseline backtest for configured pairs
- Inspect equity, drawdown, exposures, and metrics
- Validate execution invariants for modular testing


In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ensure all project-relative IO uses repository root
os.chdir(ROOT)

from src.utils.io import load_config
from src.experiments.run_baseline import run as run_baseline


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG_PATH)

backtests = run_baseline(str(CONFIG_PATH))
print("Backtests generated for:", list(backtests.keys()))


In [ ]:
metrics_path = ROOT / "reports" / "tables" / "baseline_metrics.csv"
metrics = pd.read_csv(metrics_path, index_col=0)
metrics.sort_values("sharpe", ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, bt in backtests.items():
    ax.plot(bt.index, bt["equity"], label=name, lw=1.6)
ax.set_title("Baseline equity curves")
ax.set_ylabel("Equity")
ax.grid(alpha=0.2)
ax.legend()
plt.show()


In [ ]:
for name, bt in backtests.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(bt.index, bt["drawdown"], color="crimson", lw=1.3)
    axes[0].fill_between(bt.index, bt["drawdown"], 0, color="salmon", alpha=0.35)
    axes[0].set_title(f"{name}: drawdown")
    axes[0].grid(alpha=0.2)

    axes[1].plot(bt.index, bt["gross"], label="gross", lw=1.2)
    axes[1].plot(bt.index, bt["net"], label="net", lw=1.2)
    axes[1].set_title(f"{name}: exposure")
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


In [ ]:
max_gross = float(cfg["strategy"].get("max_gross", 1.5))

for name, bt in backtests.items():
    assert bt["equity"].notna().all(), f"{name}: NaN in equity"
    assert (bt["equity"] > 0).all(), f"{name}: non-positive equity"
    assert (bt["turnover"] >= 0).all(), f"{name}: negative turnover"
    assert (bt["gross"] <= max_gross + 1e-8).all(), f"{name}: max_gross violated"

print("Baseline invariants passed.")
